# Harry Potter RAG — Lab 3: Evaluation

Continuing from [Lab 2: Evaluation Prep](./lab2-evaluation-prep.ipynb), which produced
the labelled set at `labs/eval/qa_candidates.jsonl`.

This lab measures the pipeline from Lab 1: it loads the labelled questions, runs the
retrieval and generation stages, and puts numbers behind every design choice.

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Goal of Lab 3: RAG Evaluation & Ablation Matrix

We take the baseline pipeline and measure how different design choices actually move
the numbers. Instead of treating RAG like a black box, we split evaluation into two
separate questions:

1. Did we retrieve the right context?
2. Did the LLM generate a correct, grounded answer?

### What we're testing

1. **Ground-truth evaluation** — load the labelled Q&A set: questions with exact
   evidence snippets, multi-chapter questions, and out-of-scope queries that test
   refusal behaviour.

2. **Ablation grid** — an automated matrix comparison across the main knobs:
    - **Chunking:** fixed-size (256 vs. 512 tokens) vs. chapter-aware boundaries.
    - **Retrieval:** dense (Titan V2 embeddings), sparse (BM25) and hybrid (Reciprocal Rank Fusion).

3. **Separating the failures**
    - **Retriever metrics:** `Hit@k`, `Recall@k` and `MRR` — did the labelled evidence make the context window?
    - **Generator metrics:** an LLM-as-a-Judge for answer correctness and faithfulness, plus false-abstention rate.

4. **Results leaderboard** — every run logs to `eval/results.csv`, so architectural
   choices are backed by numbers rather than vibes.

## Preconditions

- **Lab 1 pipeline** — embeddings, the Chroma collection, BM25, and the `ask()` loop.
- **Lab 2 output** — `labs/eval/qa_candidates.jsonl`. Filter on `locatable` and treat
  `evidence_chunk_ids` as the retrieval label. Lab 3 never regenerates it.

## Build order

1. Load the labelled set and the chunk index.
2. Build the config matrix: chunk size x chunking strategy x retriever.
3. Retrieve, then score `Hit@k` / `Recall@k` / `MRR` against `evidence_chunk_ids`.
4. Score generation: `EM` / `F1`, plus a judge for correctness and faithfulness.
5. Emit one row per question to `labs/eval/results.csv`.

## Import dependencies

In [6]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import csv
import collections

## Reusing functions from previous lab

In [5]:
def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))

class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:

                # Process bar
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))

chroma_client = chromadb.PersistentClient(path=str(LABS / ".chroma"))

## Load the evaluation set

In [7]:
# Labelled evaluation set produced by Lab 2 (Evaluation Prep).

def repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p


EVAL = repo_root() / "labs" / "eval"
rows = [json.loads(line) for line in open(EVAL / "qa_candidates.jsonl", encoding="utf-8") if line.strip()]
usable = [r for r in rows if r.get("locatable")]

print(f"{len(rows)} candidates | {len(usable)} usable (locatable)")
print("types:", dict(collections.Counter(r["type"] for r in usable)))
print("chapters:", len({r["chapter"] for r in usable}))
usable[0]

69 candidates | 65 usable (locatable)
types: {'factoid': 62, 'multi-hop': 3}
chapters: 17


{'id': 'cand-hp1-01-01',
 'book': 1,
 'chapter': 1,
 'chapter_name': 'Chapter 1 - The Boy Who Lived',
 'type': 'factoid',
 'question': 'What is the name of the firm where Mr. Dursley works?',
 'answer': 'Grunnings',
 'evidence': 'Mr. Dursley was the director of a firm called Grunnings, which made drills.',
 'evidence_chunk_ids': ['hp1-ch01-0001'],
 'n_spans': 1,
 'locatable': True,
 'distinctive': True,
 'unmatched_spans': [],
 'status': 'draft'}